In [21]:
# Required libraries
import regex as re
import json
import os
import base64
import unicodedata

### Configuration

In [22]:
# Base vocabulary: 256 possible byte values
BASE_VOCAB_SIZE = 256

# Total vocabulary we want after BPE training
vocab_size = 10000

# Number of BPE merges to learn
num_merges = vocab_size - BASE_VOCAB_SIZE

print("Base vocab:", BASE_VOCAB_SIZE)
print("Target vocab:", vocab_size)
print("Number of merges:", num_merges)

Base vocab: 256
Target vocab: 10000
Number of merges: 9744


In [23]:
# Load the training corpus

with open("tokenizer_training_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Number of characters:", len(text))
print("Number of bytes:", len(text.encode("utf-8")))
print("\nFirst 500 characters:\n")
print(text[:500])

Number of characters: 866250
Number of bytes: 883321

First 500 characters:

The derivative of f(x) = x^n is f'(x) = n·x^(n-1), a direct consequence of the power rule.

Notwithstanding the aforementioned limitations, the methodology employed in this study represents a meaningful advancement over previous approaches to measuring cognitive load.

Common operators: + − × ÷ = ≠ < > ≤ ≥ ± ∓

For the linear equation 3x + 7 = 22, subtracting 7 from both sides gives 3x = 15, so x = 5.

The committee's preliminary findings suggest a statistically significant correlation between e


### GPT-2 Style Pre-tokenization

In [24]:
# GPT-2 style pre-tokenization

gpt2pat = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

def pre_tokenize(text):
    chunks = re.findall(gpt2pat, text)
    return chunks

In [25]:
# Test the pre-tokenizer
pre_tokenized_text = pre_tokenize(text)

print("Number of tokens:", len(pre_tokenized_text))
print("\nFirst 10 tokens:\n")
print(pre_tokenized_text[:10])

Number of tokens: 228403

First 10 tokens:

['The', ' derivative', ' of', ' f', '(', 'x', ')', ' =', ' x', '^']


### Convert Chunks to UTF-8 Bytes

In [26]:
# Convert pre-tokenized chunks into UTF-8 byte tokens

def text_to_bytes(chunks):
    tokens = []

    for chunk in chunks:
        chunk_bytes = chunk.encode("utf-8")
        tokens.extend(chunk_bytes)

    return tokens

In [27]:
# Convert the complete corpus into byte-level tokens
text_to_bytes_tokens = text_to_bytes(pre_tokenized_text)

print("Number of bytes:", len(text_to_bytes_tokens))
print("\nFirst 10 tokens:\n")
print(text_to_bytes_tokens[:10])

Number of bytes: 883321

First 10 tokens:

[84, 104, 101, 32, 100, 101, 114, 105, 118, 97]


### Count Pair Frequencies

In [28]:
# Count frequency of adjacent token pairs

def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts

### Merge Function

In [29]:
# Merge a selected pair into a new token ID
def merge(ids, pair, idx):
    newids = []
    i = 0

    while i < len(ids):

        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2

        else:
            newids.append(ids[i])
            i += 1

    return newids

### Train BPE

In [30]:
# Learn BPE merge rules
ids = list(text_to_bytes_tokens)
merges = {}

for i in range(num_merges):

    stats = get_stats(ids)

    if not stats:
        break

    pair = max(stats, key=stats.get)
    idx = BASE_VOCAB_SIZE + i

    print(f"merging {pair} into a new token")

    ids = merge(ids, pair, idx)

    merges[pair] = idx

merging (101, 32) into a new token
merging (115, 32) into a new token
merging (44, 32) into a new token
merging (105, 110) into a new token
merging (101, 114) into a new token
merging (116, 104) into a new token
merging (100, 32) into a new token
merging (10, 10) into a new token
merging (116, 32) into a new token
merging (111, 110) into a new token
merging (114, 101) into a new token
merging (101, 110) into a new token
merging (32, 32) into a new token
merging (46, 263) into a new token
merging (97, 110) into a new token
merging (121, 32) into a new token
merging (97, 108) into a new token
merging (116, 105) into a new token
merging (111, 114) into a new token
merging (97, 114) into a new token
merging (261, 256) into a new token
merging (101, 262) into a new token
merging (32, 226) into a new token
merging (259, 103) into a new token
merging (111, 117) into a new token
merging (97, 116) into a new token
merging (61, 32) into a new token
merging (102, 32) into a new token
merging (115

In [31]:
print("Total merges:", len(merges))
print("Expected merges:", vocab_size - 256)
print("Final vocabulary size:", 256 + len(merges))

Total merges: 9744
Expected merges: 9744
Final vocabulary size: 10000


### Build Vocabulary

In [32]:
# Build vocabulary from base bytes and learned merges
vocab = {idx: bytes([idx]) for idx in range(256)}

for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print("Vocabulary size:", len(vocab))

Vocabulary size: 10000


In [33]:
# Check a few learned tokens
for idx in range(256, min(266, len(vocab))):
    print(idx, vocab[idx])

256 b'e '
257 b's '
258 b', '
259 b'in'
260 b'er'
261 b'th'
262 b'd '
263 b'\n\n'
264 b't '
265 b'on'


In [34]:
# Save tokenizer files
import base64

os.makedirs("my_tokenizer", exist_ok=True)

merge_data = [
    [p0, p1, idx]
    for (p0, p1), idx in merges.items()
]

with open("my_tokenizer/merges.json", "w", encoding="utf-8") as f:
    json.dump(merge_data, f, indent=2)

vocab_data = {
    str(idx): base64.b64encode(token_bytes).decode("ascii")
    for idx, token_bytes in vocab.items()
}

with open("my_tokenizer/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_data, f, indent=2)

config = {
    "tokenizer_type": "ByteLevelBPE",
    "vocab_size": len(vocab),
    "base_vocab_size": 256,
    "num_merges": len(merges),
    "encoding": "utf-8"
}

with open("my_tokenizer/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Tokenizer saved successfully.")

Tokenizer saved successfully.


### Load Tokenizer

In [35]:
# Load the saved tokenizer
def load_tokenizer(path="my_tokenizer"):

    with open(f"{path}/merges.json", "r", encoding="utf-8") as f:
        merge_data = json.load(f)

    merges = {
        (p0, p1): idx
        for p0, p1, idx in merge_data
    }

    with open(f"{path}/vocab.json", "r", encoding="utf-8") as f:
        vocab_data = json.load(f)

    vocab = {
        int(idx): base64.b64decode(token_bytes)
        for idx, token_bytes in vocab_data.items()
    }

    with open(f"{path}/config.json", "r", encoding="utf-8") as f:
        config = json.load(f)

    return merges, vocab, config


loaded_merges, loaded_vocab, config = load_tokenizer()

print("Loaded vocabulary:", len(loaded_vocab))
print("Loaded merges:", len(loaded_merges))

Loaded vocabulary: 10000
Loaded merges: 9744


### Encode Function

In [36]:
# Encode text into token IDs using the learned BPE merge rules
def encode(text):
    chunks = pre_tokenize(text)
    tokens = text_to_bytes(chunks)

    while len(tokens) >= 2:

        stats = get_stats(tokens)

        # Select the highest-priority learned merge
        pair = min(
            stats,
            key=lambda p: loaded_merges.get(p, float("inf"))
        )

        # Stop when no learned pair is available
        if pair not in loaded_merges:
            break

        idx = loaded_merges[pair]
        tokens = merge(tokens, pair, idx)

    return tokens

In [37]:
# Test encoding a sample sentence
sample_text = "The derivative of f(x) = x^2"

encoded = encode(sample_text)

print("Original text:")
print(sample_text)

print("\nToken IDs:")
print(encoded)

print("\nNumber of token IDs:", len(encoded))

Original text:
The derivative of f(x) = x^2

Token IDs:
[296, 4320, 691]

Number of token IDs: 3


### Decode Function

In [38]:
# Decode token IDs back into the original text
def decode(ids):
    tokens = []

    for idx in ids:
        tokens.append(loaded_vocab[idx])

    byte_data = b"".join(tokens)

    return byte_data.decode("utf-8", errors="replace")

In [39]:
# Test decoding the encoded tokens
decoded = decode(encoded)

print("Original:")
print(sample_text)

print("\nEncoded:")
print(encoded)

print("\nDecoded:")
print(decoded)

Original:
The derivative of f(x) = x^2

Encoded:
[296, 4320, 691]

Decoded:
The derivative of f(x) = x^2


In [40]:
# Install tiktoken in the same Python environment used by this notebook
import sys

!{sys.executable} -m pip install tiktoken --timeout 120 --retries 10


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
# Compare different tokenizers
import tiktoken

# GPT-4-era tokenizer
gpt4_enc = tiktoken.get_encoding("cl100k_base")

# GPT-2 tokenizer
gpt2_enc = tiktoken.get_encoding("gpt2")

# Test text
test_text = "안녕하세요 👋 (hello in Korean!)"

# Encode with each tokenizer
custom_tokens = encode(test_text)
gpt2_tokens = gpt2_enc.encode(test_text)
gpt4_tokens = gpt4_enc.encode(test_text)

print("Text:")
print(test_text)

print("\n================ CUSTOM TOKENIZER ================")
print(custom_tokens)
print("Number of tokens:", len(custom_tokens))

print("\n================ GPT-2 ================")
print(gpt2_tokens)
print("Number of tokens:", len(gpt2_tokens))

print("\n================ GPT-4 / cl100k_base ================")
print(gpt4_tokens)
print("Number of tokens:", len(gpt4_tokens))

Text:
안녕하세요 👋 (hello in Korean!)

================ CUSTOM TOKENIZER ================
[236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 935, 240, 159, 145, 139, 884, 104, 1154, 287, 307, 75, 524, 270, 33, 41]
Number of tokens: 29

================ GPT-2 ================
[168, 243, 230, 167, 227, 243, 47991, 246, 168, 226, 116, 168, 248, 242, 50169, 233, 357, 31373, 287, 6983, 8133]
Number of tokens: 21

================ GPT-4 / cl100k_base ================
[31495, 230, 75265, 243, 92245, 62904, 233, 320, 15339, 304, 16526, 16715]
Number of tokens: 12


In [42]:
# Decode the tokens back into text
print("\n================ DECODE TEST ================")

print("Custom:")
print(decode(custom_tokens))

print("\nGPT-2:")
print(gpt2_enc.decode(gpt2_tokens))

print("\nGPT-4 / cl100k_base:")
print(gpt4_enc.decode(gpt4_tokens))


================ DECODE TEST ================
Custom:
안녕하세요 👋 (hello in Korean!)

GPT-2:
안녕하세요 👋 (hello in Korean!)

GPT-4 / cl100k_base:
안녕하세요 👋 (hello in Korean!)


In [43]:
# Test different types of text
test_cases = [
    "Hello world!",
    "The derivative of f(x) = x²",
    "안녕하세요 👋",
    "नमस्ते दुनिया 🌍",
    "こんにちは",
    "machine learning",
    "transformer architecture",
    "1234567890",
    "hello@example.com"
]

for text in test_cases:

    custom = encode(text)
    gpt2 = gpt2_enc.encode(text)
    gpt4 = gpt4_enc.encode(text)

    print("\nText:", text)
    print("Custom:", len(custom), custom)
    print("GPT-2 :", len(gpt2), gpt2)
    print("GPT-4 :", len(gpt4), gpt4)


Text: Hello world!
Custom: 7 [72, 1154, 287, 660, 108, 100, 33]
GPT-2 : 3 [15496, 995, 0]
GPT-4 : 3 [9906, 1917, 0]

Text: The derivative of f(x) = x²
Custom: 3 [296, 4320, 964]
GPT-2 : 10 [464, 27255, 286, 277, 7, 87, 8, 796, 2124, 31185]
GPT-4 : 9 [791, 32905, 315, 282, 2120, 8, 284, 865, 30556]

Text: 안녕하세요 👋
Custom: 19 [236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 935, 240, 159, 145, 139]
GPT-2 : 16 [168, 243, 230, 167, 227, 243, 47991, 246, 168, 226, 116, 168, 248, 242, 50169, 233]
GPT-4 : 7 [31495, 230, 75265, 243, 92245, 62904, 233]

Text: नमस्ते दुनिया 🌍
Custom: 42 [224, 164, 168, 224, 164, 174, 224, 164, 184, 224, 165, 141, 224, 164, 164, 224, 165, 135, 32, 224, 164, 166, 224, 165, 129, 224, 164, 168, 224, 164, 191, 224, 164, 175, 224, 164, 190, 32, 240, 159, 140, 141]
GPT-2 : 26 [11976, 101, 11976, 106, 11976, 116, 24231, 235, 11976, 97, 24231, 229, 28225, 99, 24231, 223, 11976, 101, 11976, 123, 11976, 107, 48077, 12520, 234, 235]
GPT-4 : 16 [61196, 

In [44]:
encode("I AM Raj @!$%^&*():")

[453, 65, 3033, 82, 97, 106, 32, 64, 33, 36, 37, 94, 38, 42, 40, 41, 58]

In [45]:
decode([453, 65, 3033, 82, 97, 106, 32, 64, 33, 36, 37, 94, 38, 42, 40, 41, 58])

'I AM Raj @!$%^&*():'

In [46]:
import pickle, regex as re

class CustomTokenizer:
    def __init__(self, merges, vocab):
        self.merges = merges
        self.vocab = vocab
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    def encode(self, text):
        chunks = re.findall(self.pat, text)
        tokens = [b for c in chunks for b in c.encode("utf-8")]
        while len(tokens) >= 2:
            stats = {p: 0 for p in zip(tokens, tokens[1:])}
            for p in zip(tokens, tokens[1:]): stats[p] += 1
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            idx = self.merges[pair]
            newids, i = [], 0
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                    newids.append(idx); i += 2
                else:
                    newids.append(tokens[i]); i += 1
            tokens = newids
        return tokens

    def get_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") for i in ids]

# Model export to .pkl
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(CustomTokenizer(merges, vocab), f)

In [47]:
decode([743, 517, 256, 392, 391, 281, 2601, 104, 345, 528, 305, 450, 391, 97, 119, 10])

'my name  is shatrudhan prasad shaw\n'